# Aarogya — Extensive Fine-tuning of Gemma 4 / 3 (Unsloth + QLoRA)

**Hackathon:** Gemma 4 Good · Special Track: **Unsloth** ($10K)

**Hardware:** Kaggle GPU **T4 × 2** · Internet **ON**

**Runtime:** ~2-3 hours for 1500 steps on 5000 samples

Auto-falls-back to Gemma 3 4B if Unsloth hasn't published a Gemma 4 4B build yet.

## 1. Install Unsloth (uses prebuilt wheels — no source compilation)

In [ ]:
# ═══════════════════════════════════════════════════
# CELL 1: Install Unsloth + dependencies
# Works on Kaggle Python 3.12 + T4/P100/A100
# ═══════════════════════════════════════════════════
import subprocess, sys

def run(cmd):
    """Run shell command, print output, raise on failure."""
    print(f"Running: {cmd}")
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.stdout: print(result.stdout[-3000:])  # last 3KB to avoid log flood
    if result.returncode != 0:
        print("STDERR:", result.stderr[-2000:])
        raise RuntimeError(f"Command failed: {cmd}")
    return result

# Primary install: PyPI unsloth (handles Python 3.12 automatically)
run(f"{sys.executable} -m pip install unsloth -q")
run(f"{sys.executable} -m pip install --no-deps trl peft accelerate bitsandbytes -q")
run(f"{sys.executable} -m pip install datasets -q")
print("\n✅ All packages installed.")


In [ ]:
# ═══════════════════════════════════════════════════
# CELL 2: Verify installation — STOP HERE if this fails
# ═══════════════════════════════════════════════════
import torch

print(f"Python: {__import__("sys").version}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU count: {torch.cuda.device_count()}")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)} | "
              f"VRAM: {torch.cuda.get_device_properties(i).total_memory // 1024**3}GB")

try:
    from unsloth import FastLanguageModel
    print("\n✅ Unsloth imported successfully!")
except ImportError as e:
    print(f"\n❌ Unsloth import FAILED: {e}")
    print("Re-run Cell 1 or check pip output for errors.")
    raise


## 2. Clone the Aarogya repo + build dataset

In [ ]:
# ═══════════════════════════════════════════════════
# CELL 3: Clone Aarogya repo + build extensive dataset
# ═══════════════════════════════════════════════════
import os, subprocess, sys

REPO_DIR = "/kaggle/working/aarogya"
FT_DIR   = f"{REPO_DIR}/finetuning"

if not os.path.exists(REPO_DIR):
    print("Cloning Aarogya repo...")
    subprocess.run(
        ["git", "clone", "--depth=1",
         "https://github.com/shabdkumar3/aarogya.git", REPO_DIR],
        check=True
    )
else:
    print("Repo exists — pulling latest...")
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=False)

os.chdir(FT_DIR)
print(f"Working dir: {os.getcwd()}")

# Install datasets if not already there
subprocess.run([sys.executable, "-m", "pip", "install", "datasets", "-q"], check=False)

print("\nBuilding dataset (5 sources, ~15K train + 2K val)...")
result = subprocess.run([sys.executable, "prepare_dataset.py"],
                        capture_output=False, text=True)
if result.returncode != 0:
    raise RuntimeError("Dataset preparation failed — check output above.")

# Verify files exist and have content
for fname in ["train_data.jsonl", "val_data.jsonl"]:
    if not os.path.exists(fname) or os.path.getsize(fname) == 0:
        raise RuntimeError(f"Missing or empty: {fname}")
    count = sum(1 for _ in open(fname))
    print(f"  {fname}: {count} samples")

print("\n✅ Dataset ready!")


## 3. Pick the best Gemma model available on Unsloth (Gemma 4 → Gemma 3 fallback)

In [ ]:
# ═══════════════════════════════════════════════════
# CELL 4: Auto-select best available Gemma model
# ═══════════════════════════════════════════════════
from huggingface_hub import HfApi

# Priority order: Gemma 4 first, Gemma 3 fallback
CANDIDATES = [
    "unsloth/gemma-4-4b-it-unsloth-bnb-4bit",
    "unsloth/gemma-4-4b-it",
    "unsloth/gemma-3-4b-it-bnb-4bit",
    "unsloth/gemma-3-4b-it",
    "unsloth/gemma-2-2b-it-bnb-4bit",  # final fallback
]

api = HfApi()
MODEL_NAME = None

for candidate in CANDIDATES:
    try:
        api.repo_info(candidate, repo_type="model")
        MODEL_NAME = candidate
        print(f"✅ Using model: {MODEL_NAME}")
        break
    except Exception:
        print(f"  Not available: {candidate}")

if MODEL_NAME is None:
    raise RuntimeError("No Gemma model found on HuggingFace. Check internet connection and HF token.")

print(f"\nModel selected: {MODEL_NAME}")


## 4. Load model with QLoRA (4-bit)

In [ ]:
# ═══════════════════════════════════════════════════
# CELL 5: Load model + apply QLoRA adapters
# ═══════════════════════════════════════════════════
from unsloth import FastLanguageModel
import torch

MAX_SEQ_LENGTH = 1024

print(f"Loading {MODEL_NAME}...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name      = MODEL_NAME,
    max_seq_length  = MAX_SEQ_LENGTH,
    dtype           = None,          # auto-detect bf16/fp16
    load_in_4bit    = True,
)
print("Base model loaded.")

model = FastLanguageModel.get_peft_model(
    model,
    r                   = 32,
    lora_alpha          = 64,
    lora_dropout        = 0.05,
    bias                = "none",
    use_gradient_checkpointing = "unsloth",
    random_state        = 42,
    target_modules      = [
        "q_proj","k_proj","v_proj","o_proj",
        "gate_proj","up_proj","down_proj",
    ],
)

model.print_trainable_parameters()
print("\n✅ Model ready for training!")


## 5. Train (1500 steps · ~2-3h on T4×2)

In [ ]:
# ═══════════════════════════════════════════════════
# CELL 6: Fine-tune — 2000 steps (~3-4h on T4x2)
# ═══════════════════════════════════════════════════
import os, torch
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments

# Ensure we are in finetuning dir
os.chdir("/kaggle/working/aarogya/finetuning")

import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
import torch
torch.cuda.empty_cache()
print(f"Free VRAM before training: {(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated()) / 1024**3:.1f} GB")
print("Loading datasets...")
try:
    train_ds = load_dataset("json", data_files="train_data.jsonl", split="train")
    val_ds   = load_dataset("json", data_files="val_data.jsonl",   split="train")
except Exception as e:
    raise RuntimeError(f"Failed to load datasets: {e}\nRun Cell 3 first.")

print(f"Train: {len(train_ds)} samples | Val: {len(val_ds)} samples")

USE_BF16 = torch.cuda.is_bf16_supported()
print(f"Using: {"bf16" if USE_BF16 else "fp16"}")

trainer = SFTTrainer(
    model              = model,
    tokenizer          = tokenizer,
    train_dataset      = train_ds,
    eval_dataset       = val_ds,
    dataset_text_field = "text",
    max_seq_length     = MAX_SEQ_LENGTH,
    dataset_num_proc   = 2,
    args = TrainingArguments(
        output_dir                  = "/kaggle/working/checkpoints",
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 16,  # effective batch = 16 (batch=1 x16)
        warmup_steps                = 100,
        max_steps                   = 2000,
        learning_rate               = 2e-4,
        fp16                        = not USE_BF16,
        bf16                        = USE_BF16,
        logging_steps               = 50,
        eval_strategy         = "steps",
        eval_steps                  = 200,
        save_strategy               = "steps",
        save_steps                  = 500,
        save_total_limit            = 3,
        optim                       = "adamw_8bit",
        weight_decay                = 0.01,
        lr_scheduler_type           = "cosine",
        seed                        = 42,
        report_to                   = "none",
        dataloader_num_workers      = 0,    # avoid multiprocessing issues on Kaggle
    ),
)

print("\nStarting training — 2000 steps...")
print("Expected time: ~3-4 hours on T4x2")
print("Logs will appear every 50 steps\n")

try:
    stats = trainer.train()
    print(f"\n✅ Training complete!")
    print(f"   Final loss: {stats.training_loss:.4f}")
    print(f"   Total steps: {stats.global_step}")
except KeyboardInterrupt:
    print("\n⚠️ Training interrupted by user. Saving current checkpoint...")
except Exception as e:
    print(f"\n❌ Training error: {e}")
    print("Attempting to save partial model...")
    raise


## 6. Save LoRA adapter

In [ ]:
# ═══════════════════════════════════════════════════
# CELL 7: Save LoRA adapter
# ═══════════════════════════════════════════════════
import os

SAVE_DIR = "/kaggle/working/aarogya-lora"
os.makedirs(SAVE_DIR, exist_ok=True)

print(f"Saving LoRA adapter to {SAVE_DIR}...")
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

# Save training metadata
import json
metadata = {
    "base_model": MODEL_NAME,
    "lora_rank": 32,
    "lora_alpha": 64,
    "max_seq_length": MAX_SEQ_LENGTH,
    "training_steps": 2000,
    "dataset": "multi-source: ChatDoctor + Medalpaca + MediQA + MedQuad + Synthetic-Hindi",
    "train_samples": len(train_ds),
    "task": "Medical triage JSON generation for ASHA workers, rural India",
}
with open(f"{SAVE_DIR}/training_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

files = os.listdir(SAVE_DIR)
print(f"\n✅ Saved {len(files)} files:")
for fn in sorted(files):
    size = os.path.getsize(f"{SAVE_DIR}/{fn}")
    print(f"   {fn}: {size/1024:.1f} KB")


## 7. Export merged GGUF for Ollama (optional but nice for Ollama track)

In [ ]:
# ═══════════════════════════════════════════════════
# CELL 8: Export merged GGUF (optional — for Ollama)
# ═══════════════════════════════════════════════════
GGUF_DIR = "/kaggle/working/aarogya-gguf"

try:
    import os
    os.makedirs(GGUF_DIR, exist_ok=True)
    print("Exporting GGUF (q4_k_m quantization)...")
    print("This may take 10-20 minutes...")
    model.save_pretrained_gguf(
        GGUF_DIR,
        tokenizer,
        quantization_method="q4_k_m",
    )
    files = os.listdir(GGUF_DIR)
    print(f"\n✅ GGUF exported: {files}")
except Exception as e:
    print(f"\n⚠️ GGUF export skipped (optional): {e}")
    print("This is fine — LoRA adapter in Cell 7 is the primary output.")


## 8. Sample inference — fine-tuned vs base (for the writeup)

In [ ]:
# ═══════════════════════════════════════════════════
# CELL 9: Test inference — fine-tuned model
# ═══════════════════════════════════════════════════
import torch, json
from unsloth import FastLanguageModel

FastLanguageModel.for_inference(model)

SYSTEM = (
    "You are Aarogya, a medical triage assistant for ASHA workers in rural India. "
    "Output ONLY valid JSON triage with keys: conditions, urgency, next_steps, red_flags, asha_note."
)

TEST_CASES = [
    "Bachcha 2 din se paani-wala dast, kuch khaya nahi, aankhein andar dhans gayi",
    "Patient high fever 102F for 3 days, severe headache, stiff neck",
    "Aurat 8 mahine pregnant, haath-paon mein sujan, aankhon ke aage andhera",
    "Child 3 years, MUAC 10.5cm, very weak, bilateral pitting edema",
    "Saanp ne pair mein kaata, sujan aa rahi hai, ulti ho rahi hai",
]

print("Running inference on 5 test cases...\n")
print("=" * 60)

for i, case in enumerate(TEST_CASES, 1):
    prompt = (
        f"<start_of_turn>user\n{SYSTEM}\n\n"
        f"Symptoms: {case}<end_of_turn>\n"
        f"<start_of_turn>model\n"
    )
    try:
        inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
        with torch.no_grad():
            out = model.generate(
                **inputs,
                max_new_tokens    = 300,
                temperature       = 0.1,
                do_sample         = True,
                pad_token_id      = tokenizer.eos_token_id,
            )
        response = tokenizer.decode(
            out[0][inputs.input_ids.shape[1]:],
            skip_special_tokens=True
        ).strip()

        # Try to parse as JSON to verify quality
        try:
            parsed = json.loads(response)
            urgency = parsed.get("urgency", "?")
            status  = f"✅ Valid JSON | Urgency: {urgency}"
        except Exception:
            status = "⚠️ Response not valid JSON (model may need more steps)"

        print(f"Case {i}: {case[:55]}...")
        print(f"Status: {status}")
        print(f"Response: {response[:300]}")
        print("-" * 60)

    except Exception as e:
        print(f"Case {i} error: {e}")
        print("-" * 60)

print("\n✅ Inference test complete!")
print(f"Model saved at: /kaggle/working/aarogya-lora")
